In [2]:
import datetime
print(f'Notebook last run (end to end):, {datetime.datetime.now()}')

Notebook last run (end to end):, 2026-08-08 18:11:03.687245


In [ ]:
# Get data (10% of labels)
import zipfile

# Download data
# wget https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip

# Unzip the downloaded file
zip_ref = zipfile.ZipFile("10_food_classes_10_percent.zip", "r")
zip_ref.extractall()
zip_ref.close()

--2026-08-08 21:31:56--  https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip
Loaded CA certificate '/etc/ssl/certs/ca-certificates.crt'
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.68.207, 74.125.130.207, 74.125.200.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.68.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 168546183 (161M) [application/zip]
Saving to: ‘10_food_classes_10_percent.zip’

10_food_classes_10_ 100%[===================>] 160.74M  11.1MB/s    in 18s     

2026-08-08 21:32:15 (8.70 MB/s) - ‘10_food_classes_10_percent.zip’ saved [168546183/168546183]



In [5]:
import os 


for dirpath, dirnames, filenames in os.walk("10_food_classes_10_percent"):
    print(f"There are {len(dirnames)} directories and {len(dirnames)} images in '{dirpath}'.")

There are 2 directories and 2 images in '10_food_classes_10_percent'.
There are 10 directories and 10 images in '10_food_classes_10_percent/train'.
There are 0 directories and 0 images in '10_food_classes_10_percent/train/pizza'.
There are 0 directories and 0 images in '10_food_classes_10_percent/train/grilled_salmon'.
There are 0 directories and 0 images in '10_food_classes_10_percent/train/chicken_curry'.
There are 0 directories and 0 images in '10_food_classes_10_percent/train/fried_rice'.
There are 0 directories and 0 images in '10_food_classes_10_percent/train/ice_cream'.
There are 0 directories and 0 images in '10_food_classes_10_percent/train/chicken_wings'.
There are 0 directories and 0 images in '10_food_classes_10_percent/train/steak'.
There are 0 directories and 0 images in '10_food_classes_10_percent/train/ramen'.
There are 0 directories and 0 images in '10_food_classes_10_percent/train/hamburger'.
There are 0 directories and 0 images in '10_food_classes_10_percent/train/su

In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMAGE_SHAPE=(224,224)
BATCH_SIZE = 32

train_dir ="10_food_classes_10_percent/train/"
test_dir ="10_food_classes_10_percent/test/"

train_datagon = ImageDataGenerator(rescale=1/255.)
test_datagon = ImageDataGenerator(rescale=1/255.)

print('Training image')
train_data_10_percent = train_datagon.flow_from_directory(train_dir, target_size=IMAGE_SHAPE, batch_size=BATCH_SIZE, class_mode="categorical")
print('Test image')
train_data_10_percent = test_datagon.flow_from_directory(test_dir, target_size=IMAGE_SHAPE, batch_size=BATCH_SIZE, class_mode="categorical")


Training image
Found 750 images belonging to 10 classes.
Test image
Found 2500 images belonging to 10 classes.


In [8]:
import datetime
def create_tensorboard_callback(dir_name, experiment_name):
    log_dir = dir_name + "/" + experiment_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    tensorboard_callback = tf.keras.callbacks.TensorBoard(
        log_dir = log_dir
    )
    print(f"Saving TensorBoard log files to: {log_dir}")
    return tensorboard_callback


In [9]:
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras import layers

/home/tuana/Projects/Code/ML/tf/lib/python3.12/site-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [10]:
resnet_url = "https://tfhub.dev/google/imagenet/resnet_v2_50/feature_vector/4"

# Original: EfficientNetB0 feature vector (version 1)
efficientnet_url = "https://tfhub.dev/tensorflow/efficientnet/b0/feature-vector/1"

In [15]:
def create_model(model_url, num_classes=10):
    """Takes a TensorFlow Hub URL and creates a Keras model."""

    # Input layer
    inputs = tf.keras.Input(shape=IMAGE_SHAPE + (3,),name="input_layer"
    )

    # TensorFlow Hub feature extractor
    feature_extractor_layer = hub.KerasLayer(model_url,trainable=False,name="feature_extraction_layer"
    )

    # Feature extraction
    x = feature_extractor_layer(inputs)

    # Output layer
    outputs = tf.keras.layers.Dense(num_classes,activation="softmax",name="output_layer")(x)

    # Create model
    model = tf.keras.Model(inputs=inputs,outputs=outputs)

    return model

In [16]:
resnet_model = create_model(
    resnet_url,
    num_classes=train_data_10_percent.num_classes
)

resnet_model.compile(
    loss="categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"]
)

ValueError: Exception encountered when calling layer 'feature_extraction_layer' (type KerasLayer).

A KerasTensor is symbolic: it's a placeholder for a shape an a dtype. It doesn't have any actual numerical value. You cannot convert it to a NumPy array.

Call arguments received by layer 'feature_extraction_layer' (type KerasLayer):
  • inputs=<KerasTensor shape=(None, 224, 224, 3), dtype=float32, sparse=False, ragged=False, name=input_layer>
  • training=None